In [5]:
# ============================================================
# CELL 1 — Import & Load Dataset
# ============================================================
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load dataset dari CSV yang sudah dibuat di notebook 1
csv_path = '../data/processed/cases.csv'

if not os.path.exists(csv_path):
    print(f'❌ File tidak ditemukan: {csv_path}')
    print('Pastikan notebook 01 sudah dijalankan sampai selesai!')
else:
    df = pd.read_csv(csv_path)
    print(f'✅ Dataset loaded: {len(df)} kasus')
    print(f'Kolom: {df.columns.tolist()}')
    print(df.head(3))

✅ Dataset loaded: 48 kasus
Kolom: ['case_id', 'filename', 'no_perkara', 'tanggal', 'jenis_perkara', 'terdakwa', 'pasal', 'dakwaan', 'amar_putusan', 'ringkasan_fakta', 'word_count', 'text_full']
    case_id                                    filename  \
0  case_001    5_pid.sus_2026_pn_bjm_20260615001800.txt   
1  case_002  707_pid.sus_2025_pn_bjm_20260615183740.txt   
2  case_003  708_pid.sus_2025_pn_bjm_20260615184004.txt   

                                          no_perkara          tanggal  \
0  5/Pid.Sus/2026/PN BjmDEMI KEADILAN BERDASARKAN...  28 Agustus 2025   
1  707/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...     27 Juni 2025   
2  708/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...     27 Juni 2025   

                            jenis_perkara  \
0  Pidana Khusus Narkotika & Psikotropika   
1  Pidana Khusus Narkotika & Psikotropika   
2  Pidana Khusus Narkotika & Psikotropika   

                                            terdakwa  \
0                      didampingi Advok

In [6]:
# ============================================================
# CELL 2 — Buat Kolom text_for_embedding
# WAJIB dijalankan sebelum generate embeddings!
# ============================================================

def prepare_text_for_embedding(row):
    """Gabungkan field penting untuk embedding"""
    parts = []

    # Cek dan tambahkan setiap field jika ada dan tidak kosong
    for col, prefix in [
        ('ringkasan_fakta', ''),
        ('pasal',           'Pasal: '),
        ('dakwaan',         'Dakwaan: '),
        ('amar_putusan',    'Putusan: '),
    ]:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            parts.append(f"{prefix}{str(row[col]).strip()}")

    # Fallback: gunakan text_full jika ada
    if not parts:
        if 'text_full' in row.index and pd.notna(row['text_full']):
            parts.append(str(row['text_full'])[:1000])

    return ' '.join(parts) if parts else f"kasus narkotika {row.get('case_id', '')}"

# Terapkan ke seluruh DataFrame
df['text_for_embedding'] = df.apply(prepare_text_for_embedding, axis=1)

print('✅ Kolom text_for_embedding berhasil dibuat!')
print(f'Jumlah baris: {len(df)}')
print(f'\nContoh text_for_embedding[0]:\n{df["text_for_embedding"].iloc[0][:300]}')
print(f'\nContoh text_for_embedding[1]:\n{df["text_for_embedding"].iloc[1][:300]}')

✅ Kolom text_for_embedding berhasil dibuat!
Jumlah baris: 48

Contoh text_for_embedding[0]:
yang diajukan di persidangan;Setelah mendengar pembacaan Tuntutan pidana yang diajukan olehPenuntut Umum yang pada pokoknya sebagai berikut:1.Menyatakan Terdakwa Selamat Alias Amat Bin Saubari terbukti secarasah dan meyakinkan bersalah melakukan tindak pidana Tanpa hak ataumelawan hukum menawarkan u

Contoh text_for_embedding[1]:
yang diajukan di persidangan;Setelah mendengar pembacaan tuntutan pidana yang diajukan olehPenuntut Umum yang pada pokoknya sebagai berikut:1.Menyatakan terdakwa Mahmud Bin Suhaiti (Alm), terbukti bersalahmelakukan tindak pidana secara tanpa hak atau melawan hukummenawarkan untuk dijual, menjual, me


In [7]:
# ============================================================
# CELL 3 — Load Model IndoBERT
# ============================================================

print('Loading model...')
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODEL_NAME)
print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7609.18it/s]


✅ Model loaded: paraphrase-multilingual-MiniLM-L12-v2
   Device: CPU


In [8]:
# ============================================================
# CELL 4 — Generate Embeddings
# ============================================================

EMBED_PATH = '../data/processed/embeddings.npy'
os.makedirs('../data/processed', exist_ok=True)

if os.path.exists(EMBED_PATH):
    print('Loading embeddings dari file cache...')
    embeddings = np.load(EMBED_PATH)
    print(f'✅ Embeddings loaded dari cache')
else:
    print('Generating embeddings...')
    texts = df['text_for_embedding'].tolist()
    print(f'Total teks: {len(texts)}')

    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=8,
        convert_to_numpy=True
    )
    np.save(EMBED_PATH, embeddings)
    print(f'✅ Embeddings disimpan ke {EMBED_PATH}')

print(f'Shape embeddings: {embeddings.shape}')

Generating embeddings...
Total teks: 48


Batches: 100%|██████████| 6/6 [00:00<00:00,  7.00it/s]

✅ Embeddings disimpan ke ../data/processed/embeddings.npy
Shape embeddings: (48, 384)


In [10]:
# ============================================================
# CELL 5 — Fungsi Retrieval
# ============================================================

def retrieve(query: str, k: int = 5):
    """
    Mencari k kasus paling mirip dengan query
    Returns: List of dict berisi case_id, similarity, metadata
    """
    # 1. Pre-process query
    query_clean = query.strip().lower()

    # 2. Hitung vektor query
    query_embedding = model.encode(query_clean, convert_to_tensor=True)

    # 3. Hitung cosine similarity
    case_embeddings_tensor = torch.tensor(embeddings)
    similarities = util.cos_sim(query_embedding, case_embeddings_tensor)[0]

    # 4. Ambil top-k
    top_k_indices = similarities.argsort(descending=True)[:k].tolist()

    results = []
    for idx in top_k_indices:
        row = df.iloc[idx]
        results.append({
            'case_id'        : row['case_id'],
            'no_perkara'     : row.get('no_perkara', '-'),
            'similarity'     : float(similarities[idx]),
            'terdakwa'       : row.get('terdakwa', '-'),
            'pasal'          : str(row.get('pasal', '-'))[:150],
            'amar_putusan'   : str(row.get('amar_putusan', '-'))[:300],
            'ringkasan_fakta': str(row.get('ringkasan_fakta', '-'))[:300],
        })

    return results

print('✅ Fungsi retrieve() siap digunakan!')

✅ Fungsi retrieve() siap digunakan!


In [11]:
# ============================================================
# CELL 6 — Test Fungsi Retrieve
# ============================================================

query_test = (
    'terdakwa ditangkap membawa narkotika jenis sabu '
    'seberat 5 gram melanggar pasal 112 undang-undang '
    'narkotika nomor 35 tahun 2009'
)

print('🔍 Query:', query_test)
print('='*60)

hasil = retrieve(query_test, k=5)

for i, h in enumerate(hasil, 1):
    print(f'\n📌 Top-{i} (similarity = {h["similarity"]:.4f})')
    print(f'   Case ID    : {h["case_id"]}')
    print(f'   No Perkara : {h["no_perkara"]}')
    print(f'   Terdakwa   : {h["terdakwa"]}')
    print(f'   Pasal      : {h["pasal"][:80]}')
    print(f'   Amar       : {h["amar_putusan"][:100]}')

🔍 Query: terdakwa ditangkap membawa narkotika jenis sabu seberat 5 gram melanggar pasal 112 undang-undang narkotika nomor 35 tahun 2009

📌 Top-1 (similarity = 0.6631)
   Case ID    : case_003
   No Perkara : 708/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi
   Terdakwa   : MUHAMMADASRANI ALIAS AAS BIN SAHRUNI
   Pasal      : pasal 114 ayat (2) UU RI No. 35Tahun 2009 tentang Narkotika dalam dakwaan Pertam
   Amar       : perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagaiberiku

📌 Top-2 (similarity = 0.6588)
   Case ID    : case_002
   No Perkara : 707/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi
   Terdakwa   : di persidangan didampingi Penasehat Hukum bernama AgusHariyanto
   Pasal      : pasal 114 ayat (1) UU RI No. 35Tahun 2009 tentang Narkotika dalam dakwaan Pertam
   Amar       : perkara pidana denganacara pemeriksaan biasa dalam tingkat

In [12]:
# ============================================================
# CELL 7 — Fungsi Predict Outcome (Case Solution Reuse)
# ============================================================

def predict_outcome(query: str, k: int = 5) -> dict:
    """
    Prediksi solusi untuk kasus baru berdasarkan k kasus mirip
    Menggunakan weighted similarity — bobot = skor similarity
    """
    top_k = retrieve(query, k=k)

    # Kumpulkan amar putusan dari top-k
    solutions = []
    for case in top_k:
        amar = case['amar_putusan']
        if amar and amar != '-' and amar != 'nan':
            solutions.append({
                'solution' : amar,
                'weight'   : case['similarity'],
                'case_id'  : case['case_id']
            })

    # Pilih solusi dengan similarity tertinggi (weighted)
    if solutions:
        best = max(solutions, key=lambda x: x['weight'])
        predicted = best['solution']
        source_id = best['case_id']
    else:
        predicted = 'Tidak tersedia — amar putusan belum diekstrak'
        source_id = '-'

    return {
        'query'              : query[:200],
        'predicted_solution' : predicted,
        'source_case_id'     : source_id,
        'top_k_cases'        : [c['case_id'] for c in top_k],
        'top_k_similarities' : [round(c['similarity'], 4) for c in top_k],
    }

print('✅ Fungsi predict_outcome() siap!')

✅ Fungsi predict_outcome() siap!


In [13]:
# ============================================================
# CELL 8 — Demo 5 Kasus Baru
# ============================================================

os.makedirs('../data/results', exist_ok=True)

demo_queries = [
    'terdakwa kepemilikan sabu seberat 5 gram '
    'pasal 112 UU narkotika nomor 35 tahun 2009',

    'terdakwa pengedar narkotika jenis ganja '
    'seberat 1 kilogram ditangkap saat transaksi '
    'pasal 114 UU narkotika',

    'terdakwa pengguna narkotika jenis sabu '
    'ditemukan memiliki alat hisap '
    'pasal 127 UU narkotika',

    'terdakwa kurir narkotika jenis ekstasi 100 butir '
    'ditangkap di jalan pasal 114 ayat 2 UU narkotika',

    'terdakwa memiliki narkotika golongan 1 bukan tanaman '
    'seberat 0.5 gram untuk digunakan sendiri '
    'pasal 127 UU narkotika',
]

predictions = []
print('='*60)
print('DEMO PREDIKSI 5 KASUS BARU')
print('='*60)

for i, query in enumerate(demo_queries, 1):
    result = predict_outcome(query, k=5)

    predictions.append({
        'query_id'           : f'query_{i:03d}',
        'query'              : result['query'],
        'predicted_solution' : result['predicted_solution'][:300],
        'source_case_id'     : result['source_case_id'],
        'top_5_case_ids'     : ', '.join(result['top_k_cases']),
        'similarities'       : str(result['top_k_similarities']),
    })

    print(f'\n📋 Query {i}:')
    print(f'   {query[:80]}...')
    print(f'   Prediksi  : {result["predicted_solution"][:120]}')
    print(f'   Dari kasus: {result["source_case_id"]}')
    print(f'   Top-5 IDs : {result["top_k_cases"]}')
    print(f'   Similarity: {result["top_k_similarities"]}')

# Simpan hasil
df_pred = pd.DataFrame(predictions)
df_pred.to_csv('../data/results/predictions.csv',
               index=False, encoding='utf-8-sig')
print(f'\n✅ Predictions disimpan ke ../data/results/predictions.csv')
print(df_pred[['query_id','source_case_id','similarities']])

DEMO PREDIKSI 5 KASUS BARU

📋 Query 1:
   terdakwa kepemilikan sabu seberat 5 gram pasal 112 UU narkotika nomor 35 tahun 2...
   Prediksi  : perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagaiberikut dalam perkara Terd
   Dari kasus: case_003
   Top-5 IDs : ['case_003', 'case_015', 'case_002', 'case_004', 'case_043']
   Similarity: [0.6305, 0.6144, 0.6011, 0.599, 0.5955]

📋 Query 2:
   terdakwa pengedar narkotika jenis ganja seberat 1 kilogram ditangkap saat transa...
   Prediksi  : perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagaiberikut dalam perkara Terd
   Dari kasus: case_015
   Top-5 IDs : ['case_015', 'case_003', 'case_002', 'case_004', 'case_020']
   Similarity: [0.4878, 0.4838, 0.4825, 0.4734, 0.4658]

📋 Query 3:
   terdakwa pengguna narkotika jenis sabu ditemukan memiliki alat hisap pasal 127 U...
   Prediksi  : perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan

In [14]:
# ============================================================
# CELL 9 — Evaluasi Model (Tahap 5)
# ============================================================

from sklearn.metrics import classification_report

os.makedirs('../data/eval', exist_ok=True)

# Split data: 80% case base, 20% evaluasi
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f'Train : {len(df_train)} kasus')
print(f'Test  : {len(df_test)} kasus (digunakan untuk evaluasi)')

# Buat query evaluasi dari data test
queries_eval = []
for _, row in df_test.iterrows():
    queries_eval.append({
        'query_id'          : row['case_id'],
        'query_text'        : row['text_for_embedding'][:500],
        'ground_truth_id'   : row['case_id'],
    })

# Simpan query evaluasi
with open('../data/eval/queries.json', 'w', encoding='utf-8') as f:
    json.dump(queries_eval, f, ensure_ascii=False, indent=2)
print(f'✅ Query evaluasi disimpan: {len(queries_eval)} query')

Train : 38 kasus
Test  : 10 kasus (digunakan untuk evaluasi)
✅ Query evaluasi disimpan: 10 query


In [15]:
# ============================================================
# CELL 10 — Hitung Metrik Evaluasi
# ============================================================

def eval_retrieval(queries_eval, k=5):
    """Evaluasi Hit@k, Accuracy, Precision, Recall, F1"""
    y_true   = []
    y_pred   = []
    hit_at_k = []
    details  = []

    for q in tqdm(queries_eval, desc='Evaluasi'):
        query_text   = q['query_text']
        ground_truth = q['ground_truth_id']

        # Retrieve top-k
        top_k_results = retrieve(query_text, k=k)
        top_k_ids     = [r['case_id'] for r in top_k_results]
        top_k_sims    = [round(r['similarity'], 4) for r in top_k_results]

        # Hit@k
        hit = 1 if ground_truth in top_k_ids else 0
        hit_at_k.append(hit)
        y_true.append(1)
        y_pred.append(hit)

        details.append({
            'query_id'       : q['query_id'],
            'ground_truth'   : ground_truth,
            'top_k_ids'      : top_k_ids,
            'hit'            : hit,
            'best_similarity': top_k_sims[0] if top_k_sims else 0,
        })

    metrics = {
        f'Hit@{k}'  : round(np.mean(hit_at_k), 4),
        'Accuracy'  : round(accuracy_score(y_true, y_pred), 4),
        'Precision' : round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'    : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score'  : round(f1_score(y_true, y_pred, zero_division=0), 4),
    }
    return metrics, details

# Jalankan evaluasi
print('Menjalankan evaluasi...')
metrics, details = eval_retrieval(queries_eval, k=5)

print('\n' + '='*50)
print('HASIL EVALUASI MODEL IndoBERT')
print('Dataset: Pidana Khusus Narkotika & Psikotropika')
print('='*50)
for key, val in metrics.items():
    bar = '█' * int(val * 20)
    print(f'{key:<12}: {val*100:6.2f}%  {bar}')

# Simpan metrics
df_metrics = pd.DataFrame([metrics])
df_metrics.to_csv('../data/eval/retrieval_metrics.csv',
                  index=False, encoding='utf-8-sig')

# Simpan detail evaluasi
df_details = pd.DataFrame(details)
df_details.to_csv('../data/eval/prediction_metrics.csv',
                  index=False, encoding='utf-8-sig')

print(f'\n✅ Metrics disimpan ke ../data/eval/retrieval_metrics.csv')
print(f'✅ Details disimpan ke ../data/eval/prediction_metrics.csv')

Menjalankan evaluasi...


Evaluasi: 100%|██████████| 10/10 [00:00<00:00, 32.33it/s]


HASIL EVALUASI MODEL IndoBERT
Dataset: Pidana Khusus Narkotika & Psikotropika
Hit@5       :  90.00%  ██████████████████
Accuracy    :  90.00%  ██████████████████
Precision   : 100.00%  ████████████████████
Recall      :  90.00%  ██████████████████
F1-Score    :  94.74%  ██████████████████

✅ Metrics disimpan ke ../data/eval/retrieval_metrics.csv
✅ Details disimpan ke ../data/eval/prediction_metrics.csv


In [16]:
# ============================================================
# CELL 11 — Analisis Kegagalan (Error Analysis)
# ============================================================

df_details = pd.DataFrame(details)

# Kasus yang gagal (hit=0)
df_fail = df_details[df_details['hit'] == 0]
df_hit  = df_details[df_details['hit'] == 1]

print('='*50)
print('ANALISIS KEGAGALAN MODEL')
print('='*50)
print(f'Total query evaluasi : {len(df_details)}')
print(f'Berhasil (Hit=1)     : {len(df_hit)} ({len(df_hit)/len(df_details)*100:.1f}%)')
print(f'Gagal    (Hit=0)     : {len(df_fail)} ({len(df_fail)/len(df_details)*100:.1f}%)')

if len(df_fail) > 0:
    print(f'\n📌 Kasus yang GAGAL diretrieval:')
    for _, row in df_fail.iterrows():
        print(f'\n  Query ID      : {row["query_id"]}')
        print(f'  Ground Truth  : {row["ground_truth"]}')
        print(f'  Top-5 hasil   : {row["top_k_ids"]}')
        print(f'  Best Similarity: {row["best_similarity"]:.4f}')
        print(f'  → Kemungkinan penyebab: teks terlalu pendek '
              f'atau metadata tidak lengkap')

print('\n📊 Rekomendasi Perbaikan:')
print('  1. Tingkatkan kualitas preprocessing teks putusan')
print('  2. Tambah jumlah dokumen (saat ini 48, target 60+)')
print('  3. Gunakan model IndoBERT penuh jika ada GPU')
print('  4. Fine-tune model pada domain hukum narkotika')

ANALISIS KEGAGALAN MODEL
Total query evaluasi : 10
Berhasil (Hit=1)     : 9 (90.0%)
Gagal    (Hit=0)     : 1 (10.0%)

📌 Kasus yang GAGAL diretrieval:

  Query ID      : case_044
  Ground Truth  : case_044
  Top-5 hasil   : ['case_036', 'case_018', 'case_008', 'case_009', 'case_045']
  Best Similarity: 0.9084
  → Kemungkinan penyebab: teks terlalu pendek atau metadata tidak lengkap

📊 Rekomendasi Perbaikan:
  1. Tingkatkan kualitas preprocessing teks putusan
  2. Tambah jumlah dokumen (saat ini 48, target 60+)
  3. Gunakan model IndoBERT penuh jika ada GPU
  4. Fine-tune model pada domain hukum narkotika
